# Sub-Query Decomposition for Complex RAG

Multi-part questions (e.g., comparing two frameworks or requiring historical timeline correlations) demand distinct pieces of information that rarely live in a single document chunk. Query Decomposition divides complex queries into independent sub-questions, executes focused similarity retrieval for each part, and merges the context to synthesize a complete composite response.

## Workflow Architecture

<div align="center">
  <img src="workflow_query_decomposition.png" alt="Sub-Query Decomposition for Complex RAG Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. Complex Multi-Part Query"]):::startNode
    end
    subgraph Decomp["Query Decomposition"]
        Decomposer["2. Sub-Query Decomposer<br/><b>Groq gpt-oss-120b</b><br/>(Splits into Atomic Questions)"]:::decNode
        Q1["Sub-Question 1"]:::subNode
        Q2["Sub-Question 2"]:::subNode
    end
    subgraph Ret["Sub-Question Retrieval"]
        Ret1["3a. Retrieve Context for Q1"]:::vsNode
        Ret2["3b. Retrieve Context for Q2"]:::vsNode
    end
    subgraph SubAns["Sub-Answer Generation"]
        Ans1["4a. Answer Q1 with Context"]:::ansNode
        Ans2["4b. Answer Q2 with Context"]:::ansNode
    end
    subgraph Final["Final Aggregation"]
        Agg["5. Global Answer Synthesizer<br/><b>Groq gpt-oss-120b</b>"]:::aggNode
        FinalAns(["6. Complete Composite Answer"]):::endNode
    end
    Q --> Decomposer
    Decomposer --> Q1
    Decomposer --> Q2
    Q1 --> Ret1
    Q2 --> Ret2
    Ret1 --> Ans1
    Ret2 --> Ans2
    Ans1 --> Agg
    Ans2 --> Agg
    Q --> Agg
    Agg --> FinalAns
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef decNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef subNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef vsNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef ansNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef aggNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Atomic Sub-Intent Parsing**: Decomposes compound inquiries into single-topic questions.
- **Targeted Context Fetching**: Retrieves high-relevance chunks for each individual sub-intent.
- **Integrated Answer Synthesis**: Re-assembles the sub-answers into an articulate, complete final output.


In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
loader = TextLoader("langchain_crewai_dataset.txt")
raw_doc = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_doc)
chunks


c:\RAG\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\itsar\AppData\Local\Temp\ipykernel_25204\963381561.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriver = vectorstore.as_retriever(search_type="mmr", search_kwargs={'k':4, 'lambda_mult': 0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10206.81it/s]


In [4]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for
better document retrieval.
Question: "{question}"
Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

response = decomposition_chain.invoke({'question': "How does LangChain use memory and agents compared to CrewAI?"})
print(response)
# sub_questions = [q.strip("-*1234567890. ").strip() for q in response.split("\n") if q.strip()]
# print(sub_questions)

**Sub‑questions**

1. What types of memory (e.g., conversation buffers, vector stores, summary memory) does LangChain offer, and how are they integrated into LangChain workflows?  
2. How are agents designed and used in LangChain (e.g., tool‑calling agents, reasoning loops, planner agents), and what capabilities do they provide?  
3. What memory mechanisms does CrewAI provide for managing context across tasks, and how are they implemented?  
4. In what ways do CrewAI’s agents differ from LangChain’s agents in terms of architecture, tool integration, and execution flow?
['Sub‑questions', 'What types of memory (e.g., conversation buffers, vector stores, summary memory) does LangChain offer, and how are they integrated into LangChain workflows?', 'How are agents designed and used in LangChain (e.g., tool‑calling agents, reasoning loops, planner agents), and what capabilities do they provide?', 'What memory mechanisms does CrewAI provide for managing context across tasks, and how are they 

In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
Context:
{context}
Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

Decompose → Retrieve → Answer Each → Join

In [20]:
def full_query_decomposition_rag_pipeline(user_query):
    sub_qs_text = decomposition_chain.invoke({'question':user_query})
    sub_qustions = [q.strip("-*1234567890. ").strip() for q in response.split("\n") if q.strip()]

    results = []
    for subq in sub_qustions:
        docs = retriver.invoke(subq)
        result = qa_chain.invoke({'input': user_query, 'context': docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [22]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print(final_answer)

Q: Sub‑questions
A: **LangChain**

- **Memory:**  In LangChain the notion of “memory” is usually an optional component that you attach to a chain or an agent (e.g., a conversation buffer, vector store retriever, or other state‑keeping object).  The core design of LangChain is built around **prompt engineering** – reusable, nestable prompt templates, input‑variable substitution, and formatting options.  Memory is therefore a plug‑in that you add when you need a chain to remember past interactions, but it isn’t the central abstraction.

- **Agents:**  LangChain agents are essentially **prompt‑driven orchestrators** that decide which tool or chain to invoke next based on a language model’s output.  They rely heavily on the same templating and chaining primitives (Stuff, Map‑Reduce, Refine) and can be composed by re‑using prompt templates.  The focus is on **how to construct and reuse prompts**, rather than on agents exchanging data with each other.

**CrewAI**

- **Memory:**  CrewAI treat